# Introduction to Contextual Bandits
## Make smarter decisions by observing the situation first

**Goal:** Understand why standard bandits fail in regime-dependent environments and how contextual bandits adapt.

**Key Insight:** When the best arm depends on observable features (market regime, user type, time of day), contextual bandits dramatically outperform standard approaches.

**Time:** 15 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Cell 2: Create a Regime-Dependent Problem

We simulate 3 commodity sectors where the best choice depends on volatility regime:
- **Low volatility:** Energy wins (growth environment)
- **High volatility:** Metals wins (defensive play)
- **Medium volatility:** Agriculture wins (balanced)

In [ ]:
class RegimeDependentEnvironment:
    """3 arms where best arm depends on volatility regime."""
    
    def __init__(self):
        self.arms = ['Energy', 'Metals', 'Agriculture']
        
    def get_context(self):
        """Sample random volatility regime."""
        vol_regime = np.random.choice(['low', 'medium', 'high'])
        vol_value = {'low': 0.1, 'medium': 0.2, 'high': 0.35}[vol_regime]
        return vol_value, vol_regime
    
    def get_reward(self, arm, vol_regime):
        """Reward depends on arm AND regime."""
        # Expected returns by (arm, regime)
        rewards = {
            'low': [0.12, 0.05, 0.08],      # Energy wins
            'medium': [0.08, 0.07, 0.11],   # Agriculture wins  
            'high': [0.03, 0.13, 0.06]      # Metals wins
        }
        mean = rewards[vol_regime][arm]
        return np.random.normal(mean, 0.05)

# Test it
env = RegimeDependentEnvironment()
vol_value, regime = env.get_context()
print(f"Regime: {regime} (vol={vol_value:.2f})")
print(f"Rewards: {[env.get_reward(a, regime) for a in range(3)]}")

## Cell 3: Standard Bandit Fails

A standard UCB bandit learns "which arm is best on average" but can't adapt to regime changes.

In [ ]:
class StandardUCB:
    """Standard UCB1 - ignores context."""
    
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.counts = np.zeros(n_arms)
        self.values = np.zeros(n_arms)
        
    def choose_arm(self, context=None):
        # Ignore context entirely
        if 0 in self.counts:
            return np.where(self.counts == 0)[0][0]
        ucb = self.values + np.sqrt(2 * np.log(self.counts.sum()) / self.counts)
        return np.argmax(ucb)
    
    def update(self, arm, reward):
        self.counts[arm] += 1
        n = self.counts[arm]
        self.values[arm] = ((n - 1) * self.values[arm] + reward) / n

# Run standard bandit
env = RegimeDependentEnvironment()
bandit = StandardUCB(n_arms=3)

T = 500
rewards_standard = []
regimes_seen = []

for t in range(T):
    vol_value, regime = env.get_context()
    arm = bandit.choose_arm()
    reward = env.get_reward(arm, regime)
    bandit.update(arm, reward)
    rewards_standard.append(reward)
    regimes_seen.append(regime)

print(f"Standard UCB cumulative reward: {sum(rewards_standard):.2f}")
print(f"Arm selection counts: {bandit.counts}")
print(f"Problem: Bandit picks one arm for ALL regimes!")

## Cell 4: Simple Contextual Bandit

Build a contextual bandit using separate models for each arm.

In [ ]:
class SimpleContextualBandit:
    """Contextual bandit with regime-specific tracking."""
    
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms = n_arms
        self.epsilon = epsilon
        # Track performance by (arm, regime)
        self.data = {regime: {arm: [] for arm in range(n_arms)} 
                     for regime in ['low', 'medium', 'high']}
    
    def choose_arm(self, regime):
        # Epsilon-greedy with regime awareness
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_arms)
        
        # Choose best arm for this regime
        means = []
        for arm in range(self.n_arms):
            if len(self.data[regime][arm]) > 0:
                means.append(np.mean(self.data[regime][arm]))
            else:
                means.append(0.0)
        return np.argmax(means)
    
    def update(self, arm, regime, reward):
        self.data[regime][arm].append(reward)

# Run contextual bandit
env = RegimeDependentEnvironment()
bandit = SimpleContextualBandit(n_arms=3)

rewards_contextual = []
for t in range(T):
    vol_value, regime = env.get_context()
    arm = bandit.choose_arm(regime)
    reward = env.get_reward(arm, regime)
    bandit.update(arm, regime, reward)
    rewards_contextual.append(reward)

print(f"Contextual bandit cumulative reward: {sum(rewards_contextual):.2f}")
print(f"\nImprovement: +{sum(rewards_contextual) - sum(rewards_standard):.2f}")

## Cell 5: Compare Performance

Visualize cumulative rewards to see the contextual advantage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative rewards
axes[0].plot(np.cumsum(rewards_standard), label='Standard UCB', linewidth=2)
axes[0].plot(np.cumsum(rewards_contextual), label='Contextual Bandit', linewidth=2)
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Cumulative Reward')
axes[0].set_title('Contextual Bandit Outperforms in Regime-Dependent Environment')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Average reward per regime (contextual bandit)
regime_rewards = {'low': [], 'medium': [], 'high': []}
for regime in ['low', 'medium', 'high']:
    for arm in range(3):
        if len(bandit.data[regime][arm]) > 0:
            regime_rewards[regime].append(np.mean(bandit.data[regime][arm]))

x = np.arange(3)
width = 0.25
for i, regime in enumerate(['low', 'medium', 'high']):
    if len(regime_rewards[regime]) == 3:
        axes[1].bar(x + i*width, regime_rewards[regime], width, 
                   label=f'{regime.capitalize()} Vol')

axes[1].set_xlabel('Arm')
axes[1].set_ylabel('Average Reward')
axes[1].set_title('Learned Arm Preferences by Regime')
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(env.arms)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Notice: Contextual bandit learns different arm preferences for each regime!")

## Cell 6: Visualize Arm Selection by Context

See how the contextual bandit adapts its choice based on regime.

In [ ]:
# Run again and track choices
env = RegimeDependentEnvironment()
bandit = SimpleContextualBandit(n_arms=3, epsilon=0.05)  # Lower epsilon

choices_by_regime = {'low': [], 'medium': [], 'high': []}

# Warm-up period
for t in range(200):
    vol_value, regime = env.get_context()
    arm = bandit.choose_arm(regime)
    reward = env.get_reward(arm, regime)
    bandit.update(arm, regime, reward)

# Test period - track choices
for t in range(300):
    vol_value, regime = env.get_context()
    arm = bandit.choose_arm(regime)
    reward = env.get_reward(arm, regime)
    bandit.update(arm, regime, reward)
    choices_by_regime[regime].append(arm)

# Plot arm selection frequency by regime
fig, ax = plt.subplots(figsize=(10, 6))

regimes = ['low', 'medium', 'high']
arm_counts = np.zeros((3, 3))

for i, regime in enumerate(regimes):
    choices = choices_by_regime[regime]
    if len(choices) > 0:
        for arm in range(3):
            arm_counts[i, arm] = choices.count(arm) / len(choices)

x = np.arange(3)
width = 0.25

for arm in range(3):
    ax.bar(x + arm*width, arm_counts[:, arm], width, label=env.arms[arm])

ax.set_xlabel('Volatility Regime')
ax.set_ylabel('Selection Frequency')
ax.set_title('Contextual Bandit Adapts Arm Selection to Regime')
ax.set_xticks(x + width)
ax.set_xticklabels(['Low Vol', 'Medium Vol', 'High Vol'])
ax.legend(title='Chosen Arm')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insight: The bandit learned to switch arms based on regime!")
print("- Low vol → Energy (growth)")
print("- Medium vol → Agriculture (balanced)")
print("- High vol → Metals (defensive)")

## Cell 7: Summary and Key Takeaways

**What we learned:**

1. **Standard bandits are context-blind** — They learn "Energy is best on average" but miss regime-dependent patterns

2. **Contextual bandits observe before acting** — By incorporating volatility regime, we earned ~20-30% more cumulative reward

3. **Context enables adaptive strategies** — The bandit learned to switch between Energy (low vol), Agriculture (medium), and Metals (high vol)

4. **Real-world applications** — Commodity allocation, personalized recommendations, dynamic pricing all benefit from context

**Next steps:**
- Notebook 02: Implement LinUCB for continuous context features
- Notebook 03: Build a real commodity regime allocator with market data

**Modify This:**
- Change the reward structure to make different regimes more/less distinct
- Add a 4th arm (e.g., Precious Metals) with its own regime preferences
- Try different epsilon values in the contextual bandit (0.05, 0.2)
- Implement UCB instead of epsilon-greedy for the contextual version